# Phase 3 - Step 4: parameter-efficient fine-tuning (experiments #2, #3, #5)

Same pipeline and head as experiment #1 (A2 full fine-tuning, dev macro-F1 0.5616); only the **backbone and the
way it is adapted** change. The Phase-1 weights stay FROZEN; only small new parts + the sentence head are trained.

| #   | Phase-1 model       | Method                 | What is trained in the backbone                                                              |
| --- | ------------------- | ---------------------- | -------------------------------------------------------------------------------------------- |
| 2   | B (prefix-LM)       | **LoRMA** (compulsory) | h = (I + (a/r) B A) W0 x : a low-rank matrix MULTIPLIES each frozen weight, r = 8            |
| 3   | A (plain decoder)   | **LoRA**               | h = W0 x + (a/r) B A x : a low-rank matrix is ADDED to each frozen weight, r = 8             |
| 5   | A2 (decoder + copy) | **IA^3**               | only scaling vectors: keys, values and the FFN inner activations are multiplied element-wise |

LoRA / LoRMA sit on qkv, proj, w1, w2, w3 of all 10 blocks (same as Phase 2). B and A are read causally
(Phase 2: B + LoRMA causal 0.503 > bidirectional 0.485). Adapters start at "no change" (B = 0, IA^3 vectors = 1),
so step 0 is exactly the Phase-1 model. PEFT settings: lr 5e-4 (IA^3: 3e-3, as in the IA^3 paper), up to 25
epochs, early stop after 5 epochs without dev improvement.

Each experiment: Stage A (train -> best epoch on dev) -> report -> Stage B (train+dev -> test submission) -> speed -> log.
Everything is cached per run folder in `checkpoints/phase3/`, so Run All after an interruption continues where it stopped.
Expected time: ~15-30 min per experiment (~1-1.5 h for all three).


In [1]:
import sys, json, time
from pathlib import Path
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, rr_utils, rr_models, rr_experiment
for m in (rr_utils, rr_models, rr_experiment):
    importlib.reload(m)
from rr_experiment import Experiment, compare
import rr_models as rm

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

RUNS = {
    "#2 B + LoRMA": {"phase1": "B", "method": "lorma", "rank": 8, "alpha": 16},
    "#3 A + LoRA": {"phase1": "A", "method": "lora", "rank": 8, "alpha": 16},
    "#5 A2 + IA3": {"phase1": "A2", "method": "ia3"},
}

GPU: NVIDIA GeForce RTX 4050 Laptop GPU


c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Sanity check: before training, every PEFT model gives exactly the Phase-1 backbone's output

(B = 0 for LoRA/LoRMA, vectors = 1 for IA^3). We also print how many parameters each method trains.


In [2]:
check_path = ROOT / "experiments" / "phase3" / "p3_04_peft_check.json"
if check_path.exists():
    checks = json.load(open(check_path))
    print("[cache]")
else:
    checks = {}
    x = torch.randint(0, 32000, (1, 64))
    for name, ov in RUNS.items():
        exp = Experiment(ROOT, ov)
        plain, _, _ = rm.load_phase1_backbone(ROOT / "checkpoints" / "phase1" / exp.cfg["phase1_run"])
        model = exp.build_model().cpu().float()
        with torch.no_grad():
            diff = (model.backbone(x, causal=True) - plain(x, causal=True)).abs().max().item()
        total, trainable = rm.count_params(model)
        checks[name] = {"max_difference_at_start": diff, "backbone_params_added": exp.load_info["adapter_params"],
                        "trainable_total_incl_head": trainable, "params_total": total,
                        "trainable_%": round(100 * trainable / total, 2)}
        del model, plain
    json.dump(checks, open(check_path, "w"), indent=1)
for name, c in checks.items():
    print(f"{name:14s} {c}")

experiment: B_lormar8_causal_doc | device cuda | precision torch.bfloat16
experiment: A_lorar8_causal_doc | device cuda | precision torch.bfloat16
experiment: A2_ia3_causal_doc | device cuda | precision torch.bfloat16
#2 B + LoRMA   {'max_difference_at_start': 0.0, 'backbone_params_added': 1064960, 'trainable_total_incl_head': 2945581, 'params_total': 73000621, 'trainable_%': 4.04}
#3 A + LoRA    {'max_difference_at_start': 0.0, 'backbone_params_added': 875520, 'trainable_total_incl_head': 2756141, 'params_total': 72811181, 'trainable_%': 3.79}
#5 A2 + IA3    {'max_difference_at_start': 0.0, 'backbone_params_added': 30080, 'trainable_total_incl_head': 1910701, 'params_total': 71965741, 'trainable_%': 2.66}


## 2. Experiment #2: B + LoRMA


In [3]:
exp = Experiment(ROOT, RUNS["#2 B + LoRMA"])
exp.stage_a()
exp.report()
exp.stage_b()
exp.speed()
exp.log()

experiment: B_lormar8_causal_doc | device cuda | precision torch.bfloat16
parameters: 73.00M total, 2.946M trainable | loaded {'arch': 'B_prefixlm', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 1, 'dropped_groups': ['lm_head'], 'adapter_params': 1064960}


epoch  1 | train loss 2.2782 | 49s | dev macro-F1 0.2085 (CL 0.1997, IT 0.2259) acc 0.3660  <- best


epoch  2 | train loss 1.4825 | 47s | dev macro-F1 0.4470 (CL 0.3304, IT 0.4887) acc 0.5177  <- best


epoch  3 | train loss 1.1631 | 47s | dev macro-F1 0.5059 (CL 0.4238, IT 0.4843) acc 0.5761  <- best


epoch  4 | train loss 0.9444 | 47s | dev macro-F1 0.5067 (CL 0.4360, IT 0.5239) acc 0.5668  <- best


epoch  5 | train loss 0.7835 | 47s | dev macro-F1 0.5354 (CL 0.4409, IT 0.5433) acc 0.5878  <- best


... (13 outputs left out)

epoch  6 | train loss 0.4732 | 53s


epoch  7 | train loss 0.3737 | 53s
training 'full' finished -> loaded best epoch 7


submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase3\B_lormar8_causal_doc\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 710, 'Fact': 395, 'PrecedentReliedUpon': 338, 'ArgumentPetitioner': 229, 'Statute': 173, 'RulingByLowerCourt': 130, 'ArgumentRespondent': 120, 'RulingByPresentCourt': 41, 'Issue': 24, 'None': 22, 'PrecedentNotReliedUpon': 11, 'Dissent': 4}


speed: {'tokens_per_sec': 74720, 'sentences_per_sec': 1374, 'infer_sec_dev': 1.56, 'peak_gb_infer': 0.38, 'params_total': 73000621, 'params_trainable': 2945581}
logged: B_lormar8_causal_doc | dev macro-F1 0.5727


## 3. Experiment #3: A + LoRA


In [4]:
exp = Experiment(ROOT, RUNS["#3 A + LoRA"])
exp.stage_a()
exp.report()
exp.stage_b()
exp.speed()
exp.log()

experiment: A_lorar8_causal_doc | device cuda | precision torch.bfloat16
parameters: 72.81M total, 2.756M trainable | loaded {'arch': 'A_decoder', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 1, 'dropped_groups': ['lm_head'], 'adapter_params': 875520}


epoch  1 | train loss 2.3063 | 47s | dev macro-F1 0.2064 (CL 0.2162, IT 0.2293) acc 0.3739  <- best


epoch  2 | train loss 1.5108 | 47s | dev macro-F1 0.4281 (CL 0.3193, IT 0.4620) acc 0.4949  <- best


epoch  3 | train loss 1.1457 | 47s | dev macro-F1 0.4372 (CL 0.3391, IT 0.4816) acc 0.5247  <- best


epoch  4 | train loss 0.8925 | 47s | dev macro-F1 0.5346 (CL 0.4700, IT 0.5403) acc 0.5798  <- best


epoch  5 | train loss 0.6926 | 48s | dev macro-F1 0.5189 (CL 0.4027, IT 0.5335) acc 0.5630


... (23 outputs left out)

epoch 11 | train loss 0.1457 | 53s


epoch 12 | train loss 0.1280 | 53s
training 'full' finished -> loaded best epoch 12


submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase3\A_lorar8_causal_doc\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 756, 'Fact': 418, 'PrecedentReliedUpon': 301, 'ArgumentPetitioner': 219, 'Statute': 176, 'RulingByLowerCourt': 103, 'ArgumentRespondent': 95, 'RulingByPresentCourt': 41, 'None': 25, 'PrecedentNotReliedUpon': 24, 'Dissent': 23, 'Issue': 16}


speed: {'tokens_per_sec': 73371, 'sentences_per_sec': 1349, 'infer_sec_dev': 1.59, 'peak_gb_infer': 0.38, 'params_total': 72811181, 'params_trainable': 2756141}
logged: A_lorar8_causal_doc | dev macro-F1 0.5838


## 4. Experiment #5: A2 + IA^3


In [5]:
exp = Experiment(ROOT, RUNS["#5 A2 + IA3"])
exp.stage_a()
exp.report()
exp.stage_b()
exp.speed()
exp.log()

experiment: A2_ia3_causal_doc | device cuda | precision torch.bfloat16
parameters: 71.97M total, 1.911M trainable | loaded {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head'], 'adapter_params': 30080}


epoch  1 | train loss 2.2852 | 49s | dev macro-F1 0.2255 (CL 0.2236, IT 0.2341) acc 0.3828  <- best


epoch  2 | train loss 1.5717 | 49s | dev macro-F1 0.3758 (CL 0.3172, IT 0.3615) acc 0.4542  <- best


epoch  3 | train loss 1.2725 | 49s | dev macro-F1 0.4240 (CL 0.3561, IT 0.4480) acc 0.4911  <- best


epoch  4 | train loss 1.0998 | 49s | dev macro-F1 0.4575 (CL 0.3404, IT 0.4942) acc 0.5252  <- best


epoch  5 | train loss 0.8999 | 49s | dev macro-F1 0.4333 (CL 0.3059, IT 0.5079) acc 0.5345


... (23 outputs left out)

epoch 11 | train loss 0.3085 | 55s


epoch 12 | train loss 0.2858 | 56s
training 'full' finished -> loaded best epoch 12


submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase3\A2_ia3_causal_doc\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 693, 'Fact': 431, 'PrecedentReliedUpon': 337, 'ArgumentPetitioner': 252, 'Statute': 184, 'RulingByLowerCourt': 113, 'ArgumentRespondent': 70, 'RulingByPresentCourt': 40, 'None': 23, 'PrecedentNotReliedUpon': 21, 'Dissent': 17, 'Issue': 16}


speed: {'tokens_per_sec': 72392, 'sentences_per_sec': 1331, 'infer_sec_dev': 1.61, 'peak_gb_infer': 0.38, 'params_total': 71965741, 'params_trainable': 1910701}
logged: A2_ia3_causal_doc | dev macro-F1 0.566


## 5. Comparison of all Phase-3 runs so far (dev)


In [6]:
compare(ROOT)

run                                     macro     CL     IT    acc  ep     Fact   Issue Argumen Argumen Statute Dissent Precede Precede Precede RulingB RatioOf RulingB    None
A_lorar8_causal_doc                    0.5838 0.4885 0.5939 0.6232  12    0.744   0.710   0.392   0.549   0.812   0.000   0.577   0.394   0.000   0.616   0.731   0.551   0.346
B_lormar8_causal_doc                   0.5727 0.5388 0.5286 0.6433   7    0.753   0.545   0.599   0.560   0.824   0.000   0.634   0.336   0.000   0.450   0.698   0.600   0.300
A2_ia3_causal_doc                      0.5660 0.5159 0.5572 0.6237  12    0.722   0.538   0.488   0.614   0.750   0.000   0.557   0.312   0.000   0.667   0.678   0.592   0.308
A2_full_causal_doc                     0.5616 0.4883 0.5434 0.6228   7    0.715   0.629   0.574   0.561   0.700   0.000   0.530   0.371   0.000   0.529   0.713   0.565   0.292


[('A_lorar8_causal_doc',
  0.583777731396533,
  0.4884868641898356,
  0.5939389340521662,
  0.623249299719888,
  12,
  {'Fact': 0.7441,
   'Issue': 0.7097,
   'ArgumentPetitioner': 0.3915,
   'ArgumentRespondent': 0.549,
   'Statute': 0.8117,
   'Dissent': 0.0,
   'PrecedentReliedUpon': 0.5773,
   'PrecedentNotReliedUpon': 0.3944,
   'PrecedentOverruled': 0.0,
   'RulingByLowerCourt': 0.6162,
   'RatioOfTheDecision': 0.7306,
   'RulingByPresentCourt': 0.551,
   'None': 0.3462}),
 ('B_lormar8_causal_doc',
  0.5727034187154009,
  0.5388111989899461,
  0.5286490665992661,
  0.6433239962651728,
  7,
  {'Fact': 0.7533,
   'Issue': 0.5455,
   'ArgumentPetitioner': 0.5992,
   'ArgumentRespondent': 0.5605,
   'Statute': 0.8235,
   'Dissent': 0.0,
   'PrecedentReliedUpon': 0.6338,
   'PrecedentNotReliedUpon': 0.336,
   'PrecedentOverruled': 0.0,
   'RulingByLowerCourt': 0.45,
   'RatioOfTheDecision': 0.698,
   'RulingByPresentCourt': 0.6,
   'None': 0.3}),
 ('A2_ia3_causal_doc',
  0.56596767167